# Notebook 14 — XGBoost (Version 2)

This notebook develops the locked Version 2 **bank-only** and
**bank-plus-macro** XGBoost specifications on all three purged
expanding-window folds. It uses the exact ordered lagged feature locks from
Notebook 11 and never loads the 2019 purge or final-test rows.

The exact completed Version 1 XGBoost candidate grid is reused unchanged:
12 predeclared configurations spanning 100/200/300 trees, depths 2–5,
learning rates 0.03/0.05/0.10, child weights 1/5/10, row and feature
sampling, L1/L2 regularisation and positive-class weight multipliers
0.5/1.0/2.0. Tree construction is fixed to the histogram algorithm.

Median imputation and 1st/99th-percentile winsorisation are fitted separately
within each specification and training fold and applied unchanged to
validation. Every predictor-level missing value—including limited-history,
genuine and FFIEC 051 structural missingness—is imputed before model fitting;
native XGBoost missing-value treatment is never used. The locked
`insufficient_history_5q_lag1` indicator is present in both specifications
and excluded only from winsorisation. Inputs are not standardised.

Each completed fit is checkpointed atomically. A rerun automatically resumes
only checkpoints bound to the same grid, inputs, feature order, fold rows and
preprocessing parameters. Configuration selection is validation-only and
threshold selection remains deferred to Notebook 15.


In [ ]:
# Resolve V2 handoffs and freeze the unchanged XGBoost grid

from pathlib import Path
from datetime import datetime, timezone
import gc
import hashlib
import importlib.util
import json
import os
import time

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
try:
    import xgboost as xgb
    from xgboost import XGBClassifier
except ImportError as exc:
    raise ImportError(
        "XGBoost is not installed. In Colab run `!pip install -q xgboost`, "
        "restart the runtime, and rerun Cell 1."
    ) from exc


COLAB_MYDRIVE = Path("/content/drive/MyDrive")
try:
    from google.colab import drive
except ImportError:
    drive = None

if COLAB_MYDRIVE.is_dir():
    print("Google Drive is already mounted at /content/drive.")
elif drive is not None:
    try:
        drive.mount("/content/drive", force_remount=False)
    except (ValueError, OSError) as exc:
        if COLAB_MYDRIVE.is_dir():
            print("Google Drive became available after the mount warning; continuing.")
        else:
            raise RuntimeError(
                "Google Drive could not be mounted. Complete authorisation and rerun this cell."
            ) from exc


DEFAULT_PROJECT_ROOT = Path(
    os.environ.get("DISSERTATION_PROJECT_ROOT", "/content/drive/MyDrive/Data Final")
).expanduser()
explicit_config = os.environ.get("V2_CONFIG_PATH")
CONFIG_PATH = (
    Path(explicit_config).expanduser()
    if explicit_config
    else DEFAULT_PROJECT_ROOT / "Version 2 Main Analysis" / "config.py"
)
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f"Shared Version 2 config not found: {CONFIG_PATH}")

config_spec = importlib.util.spec_from_file_location("v2_config", CONFIG_PATH)
config = importlib.util.module_from_spec(config_spec)
config_spec.loader.exec_module(config)
REQUIRED_CONFIG_FIELDS = {
    "ANALYSIS_VERSION", "VERSION_2_MAIN_TIMING_POLICY", "V2_OUTPUT_ROOT",
    "VALIDATION_FOLDS", "FINAL_TEST_DESIGN", "TARGET_HORIZON_DAYS",
    "RANDOM_SEED", "MISSING_DATA_POLICY", "HISTORY_POLICY",
}
missing_config_fields = sorted(REQUIRED_CONFIG_FIELDS.difference(dir(config)))
if missing_config_fields:
    raise AttributeError("Shared config missing: " + ", ".join(missing_config_fields))

ANALYSIS_VERSION = config.ANALYSIS_VERSION
TIMING_POLICY = dict(config.VERSION_2_MAIN_TIMING_POLICY)
VALIDATION_FOLDS = tuple(dict(item) for item in config.VALIDATION_FOLDS)
FINAL_TEST_DESIGN = dict(config.FINAL_TEST_DESIGN)
TARGET_HORIZON_DAYS = int(config.TARGET_HORIZON_DAYS)
RANDOM_SEED = int(config.RANDOM_SEED)
MISSING_DATA_POLICY = dict(config.MISSING_DATA_POLICY)
HISTORY_POLICY = dict(config.HISTORY_POLICY)
V2_OUTPUT_ROOT = Path(config.V2_OUTPUT_ROOT).expanduser()
PROJECT_ROOT = Path(getattr(config, "PROJECT_ROOT", CONFIG_PATH.parent.parent)).expanduser()
V2_ROOT = Path(getattr(config, "V2_ROOT", CONFIG_PATH.parent)).expanduser()


def quarter_end(label):
    return pd.Period(str(label), freq="Q").end_time.normalize()


def deduplicate_paths(paths):
    result = []
    for item in paths:
        item = Path(item).expanduser()
        if item not in result:
            result.append(item)
    return result


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def passed_values(series):
    return series.astype(str).str.strip().str.lower().isin({"true", "1", "pass"})


def is_relative_to(path, root):
    try:
        Path(path).resolve().relative_to(Path(root).resolve())
        return True
    except ValueError:
        return False


bundle_paths = {
    "panel": Path("09_Final_Modelling_Panel") / "v2_final_modelling_panel_2001Q2_2025Q2.parquet",
    "membership": Path("10_Chronological_Validation_Design") / "v2_chronological_row_membership.parquet",
    "period_summary": Path("10_Chronological_Validation_Design") / "v2_chronological_period_summary.csv",
    "selection_rule": Path("10_Chronological_Validation_Design") / "v2_precommitted_model_selection_rule.csv",
    "nb10_gate": Path("10_Chronological_Validation_Design") / "cell_07_final_validation_gate.csv",
    "locked_sets": Path("11_EDA_and_Final_Predictor_Selection") / "locked_predictor_sets_v2.json",
    "bank_manifest": Path("11_EDA_and_Final_Predictor_Selection") / "v2_bank_only_feature_manifest.csv",
    "full_manifest": Path("11_EDA_and_Final_Predictor_Selection") / "v2_bank_plus_macro_feature_manifest.csv",
    "nb11_gate": Path("11_EDA_and_Final_Predictor_Selection") / "cell_09_final_validation_gate.csv",
}


def bundle_complete(root):
    return all((root / rel).is_file() for rel in bundle_paths.values())


candidate_roots = deduplicate_paths([
    V2_OUTPUT_ROOT / ANALYSIS_VERSION,
    V2_OUTPUT_ROOT,
    V2_ROOT / "Notebook Outputs" / ANALYSIS_VERSION,
    V2_ROOT / "Notebook Outputs",
    PROJECT_ROOT / "Notebook Outputs" / ANALYSIS_VERSION,
    PROJECT_ROOT / "Notebook Outputs",
    DEFAULT_PROJECT_ROOT / "Notebook Outputs" / ANALYSIS_VERSION,
    DEFAULT_PROJECT_ROOT / "Notebook Outputs",
])
complete_roots = [root for root in candidate_roots if bundle_complete(root)]
if not complete_roots:
    for search_root in deduplicate_paths([
        V2_OUTPUT_ROOT,
        V2_ROOT / "Notebook Outputs",
        PROJECT_ROOT / "Notebook Outputs",
        DEFAULT_PROJECT_ROOT / "Notebook Outputs",
    ]):
        if not search_root.is_dir():
            continue
        for panel_candidate in search_root.rglob(
            "v2_final_modelling_panel_2001Q2_2025Q2.parquet"
        ):
            inferred_root = panel_candidate.parent.parent
            if bundle_complete(inferred_root) and inferred_root not in complete_roots:
                complete_roots.append(inferred_root)

if not complete_roots:
    diagnostic_lines = []
    for root in candidate_roots:
        missing = [str(rel) for rel in bundle_paths.values() if not (root / rel).is_file()]
        diagnostic_lines.append(
            f"{root}\n  missing: " + ("; ".join(missing) if missing else "none")
        )
    raise FileNotFoundError(
        "No complete Version 2 Notebook 09/10/11 handoff bundle was found. "
        "The per-root diagnostic is:\n" + "\n".join(diagnostic_lines)
    )

VERSION_OUTPUT_ROOT = complete_roots[0]
NB14_HANDOFF_ROOT = (
    VERSION_OUTPUT_ROOT
    if is_relative_to(VERSION_OUTPUT_ROOT, V2_ROOT)
    else V2_OUTPUT_ROOT / ANALYSIS_VERSION
)
NB14_OUTPUT_DIR = NB14_HANDOFF_ROOT / "14_XGBoost"
NB14_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FINAL_PANEL_PATH = VERSION_OUTPUT_ROOT / bundle_paths["panel"]
MEMBERSHIP_PATH = VERSION_OUTPUT_ROOT / bundle_paths["membership"]
PERIOD_SUMMARY_PATH = VERSION_OUTPUT_ROOT / bundle_paths["period_summary"]
MODEL_SELECTION_RULE_PATH = VERSION_OUTPUT_ROOT / bundle_paths["selection_rule"]
NB10_GATE_PATH = VERSION_OUTPUT_ROOT / bundle_paths["nb10_gate"]
LOCKED_SETS_PATH = VERSION_OUTPUT_ROOT / bundle_paths["locked_sets"]
BANK_MANIFEST_PATH = VERSION_OUTPUT_ROOT / bundle_paths["bank_manifest"]
FULL_MANIFEST_PATH = VERSION_OUTPUT_ROOT / bundle_paths["full_manifest"]
NB11_GATE_PATH = VERSION_OUTPUT_ROOT / bundle_paths["nb11_gate"]
NB13_GATE_PATH = (
    NB14_HANDOFF_ROOT / "13_Random_Forest" / "cell_07_final_validation_gate.csv"
)

IDENTIFIER_COLUMNS = ["report_date", "cert", "idrssd"]
TARGET = "failed_1yr"
FOLDS = [f"fold{int(item['fold'])}" for item in VALIDATION_FOLDS]
ROLE_COLUMNS = [f"{fold}_role" for fold in FOLDS]
HISTORY_INDICATOR = HISTORY_POLICY["indicator_model_name"]
DEVELOPMENT_START = pd.Timestamp(TIMING_POLICY["reference_calendar_start"]).normalize()
DEVELOPMENT_END = quarter_end(FINAL_TEST_DESIGN["train_end"])

# Exact completed Version 1 XGBoost candidate grid. Gamma remained fixed at zero.
XGB_CONFIGURATIONS = [
    {"config_id": "xgb_01", "n_estimators": 100, "max_depth": 2, "learning_rate": 0.10, "min_child_weight": 1,  "subsample": 1.0, "colsample_bytree": 1.0, "gamma": 0.0, "reg_lambda": 1.0,  "reg_alpha": 0.0, "positive_weight_multiplier": 0.5},
    {"config_id": "xgb_02", "n_estimators": 100, "max_depth": 3, "learning_rate": 0.10, "min_child_weight": 5,  "subsample": 0.8, "colsample_bytree": 0.8, "gamma": 0.0, "reg_lambda": 5.0,  "reg_alpha": 0.1, "positive_weight_multiplier": 1.0},
    {"config_id": "xgb_03", "n_estimators": 100, "max_depth": 4, "learning_rate": 0.05, "min_child_weight": 10, "subsample": 0.7, "colsample_bytree": 1.0, "gamma": 0.0, "reg_lambda": 10.0, "reg_alpha": 0.0, "positive_weight_multiplier": 2.0},
    {"config_id": "xgb_04", "n_estimators": 100, "max_depth": 5, "learning_rate": 0.05, "min_child_weight": 1,  "subsample": 1.0, "colsample_bytree": 0.7, "gamma": 0.0, "reg_lambda": 1.0,  "reg_alpha": 0.1, "positive_weight_multiplier": 0.5},
    {"config_id": "xgb_05", "n_estimators": 200, "max_depth": 2, "learning_rate": 0.05, "min_child_weight": 5,  "subsample": 0.7, "colsample_bytree": 0.8, "gamma": 0.0, "reg_lambda": 10.0, "reg_alpha": 0.1, "positive_weight_multiplier": 1.0},
    {"config_id": "xgb_06", "n_estimators": 200, "max_depth": 3, "learning_rate": 0.05, "min_child_weight": 10, "subsample": 1.0, "colsample_bytree": 1.0, "gamma": 0.0, "reg_lambda": 1.0,  "reg_alpha": 0.0, "positive_weight_multiplier": 2.0},
    {"config_id": "xgb_07", "n_estimators": 200, "max_depth": 4, "learning_rate": 0.03, "min_child_weight": 1,  "subsample": 0.8, "colsample_bytree": 0.7, "gamma": 0.0, "reg_lambda": 5.0,  "reg_alpha": 0.1, "positive_weight_multiplier": 0.5},
    {"config_id": "xgb_08", "n_estimators": 200, "max_depth": 5, "learning_rate": 0.03, "min_child_weight": 5,  "subsample": 1.0, "colsample_bytree": 0.8, "gamma": 0.0, "reg_lambda": 10.0, "reg_alpha": 0.0, "positive_weight_multiplier": 1.0},
    {"config_id": "xgb_09", "n_estimators": 300, "max_depth": 2, "learning_rate": 0.03, "min_child_weight": 10, "subsample": 0.8, "colsample_bytree": 1.0, "gamma": 0.0, "reg_lambda": 5.0,  "reg_alpha": 0.0, "positive_weight_multiplier": 2.0},
    {"config_id": "xgb_10", "n_estimators": 300, "max_depth": 3, "learning_rate": 0.03, "min_child_weight": 1,  "subsample": 0.7, "colsample_bytree": 0.7, "gamma": 0.0, "reg_lambda": 10.0, "reg_alpha": 0.1, "positive_weight_multiplier": 0.5},
    {"config_id": "xgb_11", "n_estimators": 300, "max_depth": 4, "learning_rate": 0.05, "min_child_weight": 5,  "subsample": 1.0, "colsample_bytree": 0.8, "gamma": 0.0, "reg_lambda": 1.0,  "reg_alpha": 0.0, "positive_weight_multiplier": 1.0},
    {"config_id": "xgb_12", "n_estimators": 300, "max_depth": 5, "learning_rate": 0.05, "min_child_weight": 10, "subsample": 0.8, "colsample_bytree": 1.0, "gamma": 0.0, "reg_lambda": 5.0, "reg_alpha": 0.1, "positive_weight_multiplier": 2.0},
]
TREE_METHOD = "hist"
OBJECTIVE = "binary:logistic"
XGB_EVAL_METRIC = "logloss"
EARLY_STOPPING_ROUNDS = None
WINSOR_LOWER_QUANTILE = 0.01
WINSOR_UPPER_QUANTILE = 0.99
DEFAULT_CLASSIFICATION_THRESHOLD = 0.50
SELECTION_TOLERANCE = 1e-6
N_JOBS = -1
SESSION_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
run_contract = {
    "analysis_version": ANALYSIS_VERSION,
    "model": "XGBClassifier",
    "grid": XGB_CONFIGURATIONS,
    "tree_method": TREE_METHOD,
    "objective": OBJECTIVE,
    "winsor_quantiles": [WINSOR_LOWER_QUANTILE, WINSOR_UPPER_QUANTILE],
    "random_seed_from_shared_config": RANDOM_SEED,
    "folds": VALIDATION_FOLDS,
}
RUN_SIGNATURE = "nb14_xgb_" + hashlib.sha256(
    json.dumps(run_contract, sort_keys=True).encode("utf-8")
).hexdigest()[:24]

PREDICTION_DIR = NB14_OUTPUT_DIR / "validation_predictions_resumable_v2"
PREPROCESSING_DIR = NB14_OUTPUT_DIR / "preprocessing_parameters_resumable_v2"
CHECKPOINT_DIR = NB14_OUTPUT_DIR / "fit_checkpoints_resumable_v2"
SESSION_DIR = NB14_OUTPUT_DIR / "sessions"
for directory in [PREDICTION_DIR, PREPROCESSING_DIR, CHECKPOINT_DIR, SESSION_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

nb10_gate = pd.read_csv(NB10_GATE_PATH)
nb11_gate = pd.read_csv(NB11_GATE_PATH)
if not NB13_GATE_PATH.is_file():
    raise FileNotFoundError(
        "Notebook 13 final PASS gate was not found at the V2-isolated handoff path: "
        + str(NB13_GATE_PATH)
    )
nb13_gate = pd.read_csv(NB13_GATE_PATH)
model_selection_rule = pd.read_csv(MODEL_SELECTION_RULE_PATH)
config_frame = pd.DataFrame(XGB_CONFIGURATIONS)

setup_checks = {
    "Analysis is the Version 2 lagged-bank main design":
        ANALYSIS_VERSION == "v2_bank_lag1_main",
    "Exact t-1 bank and macro timing remains locked":
        TIMING_POLICY.get("bank_predictor_source") == "exact_t_minus_1"
        and TIMING_POLICY.get("macro_predictor_source") == "t_minus_1",
    "Target remains anchored at t over 365 days":
        TIMING_POLICY.get("target_window") == "(t, t_plus_365_days]"
        and TARGET_HORIZON_DAYS == 365,
    "All three folds and fresh hyperparameter tuning are required":
        TIMING_POLICY.get("repeat_all_validation_folds") is True
        and TIMING_POLICY.get("retune_hyperparameters") is True
        and TIMING_POLICY.get("reuse_version_1_candidate_grids") is True
        and FOLDS == ["fold1", "fold2", "fold3"],
    "Notebook 10, 11 and 13 final gates are all PASS":
        passed_values(nb10_gate["passed"]).all()
        and passed_values(nb11_gate["passed"]).all()
        and passed_values(nb13_gate["passed"]).all(),
    "Exactly 12 unique completed-V1 configurations are locked":
        len(config_frame) == config_frame["config_id"].nunique() == 12
        and not config_frame.astype(str).duplicated().any(),
    "Every required tuning dimension varies": all(
        config_frame[column].astype(str).nunique() >= 2
        for column in [
            "n_estimators", "max_depth", "learning_rate", "min_child_weight",
            "subsample", "colsample_bytree", "reg_lambda", "reg_alpha",
            "positive_weight_multiplier",
        ]
    ),
    "Gamma remains fixed at zero as in the completed Version 1 grid":
        config_frame["gamma"].eq(0.0).all(),
    "Histogram construction and no early stopping are locked":
        TREE_METHOD == "hist" and EARLY_STOPPING_ROUNDS is None,
    "Training-fold median imputation is locked":
        MISSING_DATA_POLICY.get("strategy") == "training_sample_median"
        and MISSING_DATA_POLICY.get("fit_scope") == "training_rows_within_each_fold_only",
    "Notebook 14 writes only inside the V2 root":
        is_relative_to(NB14_OUTPUT_DIR, V2_ROOT),
    "Notebook 10 selection rule is validation-only":
        len(model_selection_rule) == 3
        and not model_selection_rule["uses_final_test"].astype(str).str.strip().str.lower().isin(
            {"true", "1", "yes"}
        ).any()
        and model_selection_rule.iloc[0]["rule"].startswith(
            "Highest unweighted mean PR-AUC"
        ),
}
notebook_14_cell_01_gate = pd.DataFrame({
    "check": setup_checks.keys(),
    "result": ["PASS" if value else "FAIL" for value in setup_checks.values()],
    "passed": setup_checks.values(),
})
tuning_contract = pd.DataFrame({
    "item": [
        "analysis version", "model", "specifications", "folds",
        "candidate configurations", "number of trees", "maximum depth",
        "learning rate", "minimum child weight", "row sampling",
        "feature sampling", "gamma", "L2 regularisation", "L1 regularisation",
        "positive-class weighting", "tree construction", "checkpointing",
        "imputation", "native missing treatment", "winsorisation",
        "standardisation", "early stopping", "primary selection",
        "first tie-break", "second tie-break", "threshold selection",
        "final-test use", "Version 1 fitted artifacts",
    ],
    "locked_rule": [
        ANALYSIS_VERSION, "XGBoost", "bank_only and bank_plus_macro",
        "three purged expanding windows", "12 predeclared configurations",
        "100, 200 or 300", "2, 3, 4 or 5", "0.03, 0.05 or 0.10",
        "1, 5 or 10", "70%, 80% or 100% of each training fold per round",
        "70%, 80% or 100% of predictors per tree", "fixed at zero",
        "1, 5 or 10", "0 or 0.1",
        "training-fold negative-to-positive ratio multiplied by 0.5, 1.0 or 2.0",
        "CPU-compatible histogram algorithm",
        "atomic checkpoint after every fit; context-valid checkpoints resume",
        "training-fold median for every predictor, including limited-history, genuine and FFIEC 051 structural missingness",
        "prohibited; fitted matrices must be entirely finite before XGBoost sees them",
        "training-fold 1st/99th percentiles; history indicator excluded",
        "not used for XGBoost", "not used; number of trees is grid-tuned",
        "highest unweighted mean validation PR-AUC",
        "higher unweighted mean validation ROC-AUC within 1e-6",
        "lower validation PR-AUC standard deviation within 1e-6",
        "deferred to Notebook 15; 0.50 metrics descriptive only",
        "prohibited", "not loaded or reused",
    ],
})
grid_output = config_frame.copy()
grid_output["tree_method"] = TREE_METHOD
grid_output["grid_source"] = "unchanged completed Version 1 XGBoost candidate grid"
notebook_14_cell_01_gate.to_csv(NB14_OUTPUT_DIR / "cell_01_setup_gate.csv", index=False)
tuning_contract.to_csv(
    NB14_OUTPUT_DIR / "cell_01_xgboost_tuning_contract.csv", index=False
)
grid_output.to_csv(
    NB14_OUTPUT_DIR / "cell_01_xgboost_configuration_grid.csv", index=False
)
(SESSION_DIR / f"{SESSION_ID}.json").write_text(
    json.dumps({
        "session_id": SESSION_ID,
        "started_utc": datetime.now(timezone.utc).isoformat(),
        "run_signature": RUN_SIGNATURE,
        "input_bundle_root": str(VERSION_OUTPUT_ROOT),
        "output_handoff_root": str(NB14_HANDOFF_ROOT),
        "random_seed_from_shared_config": RANDOM_SEED,
        "final_test_loaded": False,
        "version_1_artifacts_loaded": False,
    }, indent=2),
    encoding="utf-8",
)
print("Notebook 14 Cell 1 validation gate:")
display(notebook_14_cell_01_gate)
print("\nLocked XGBoost tuning contract:")
with pd.option_context("display.max_colwidth", None):
    display(tuning_contract)
print("\nUnchanged 12-configuration grid:")
display(grid_output)
failed = [name for name, passed in setup_checks.items() if not passed]
if failed:
    raise AssertionError("Cell 1 blocked: " + "; ".join(failed))
print("\nResolved V2 input root:", VERSION_OUTPUT_ROOT)
print("V2-isolated Notebook 14 output root:", NB14_HANDOFF_ROOT)
print("Stable checkpoint signature:", RUN_SIGNATURE)


In [ ]:
# Load development rows only and reconcile features, folds and inputs

if not notebook_14_cell_01_gate["passed"].all():
    raise AssertionError("Cell 2 blocked because Cell 1 did not pass.")

with open(LOCKED_SETS_PATH, "r", encoding="utf-8") as handle:
    locked_sets = json.load(handle)
bank_manifest = pd.read_csv(BANK_MANIFEST_PATH).sort_values("feature_order")
full_manifest = pd.read_csv(FULL_MANIFEST_PATH).sort_values("feature_order")
period_summary = pd.read_csv(PERIOD_SUMMARY_PATH, parse_dates=["start", "end"])

BANK_ONLY_PREDICTORS = bank_manifest["version_2_feature"].astype(str).tolist()
BANK_PLUS_MACRO_PREDICTORS = full_manifest["version_2_feature"].astype(str).tolist()
SPECIFICATIONS = {
    "bank_only": BANK_ONLY_PREDICTORS,
    "bank_plus_macro": BANK_PLUS_MACRO_PREDICTORS,
}
all_predictors = list(dict.fromkeys(BANK_PLUS_MACRO_PREDICTORS))
panel_columns = IDENTIFIER_COLUMNS + all_predictors + [TARGET]
membership_columns = [
    "panel_row_id", *IDENTIFIER_COLUMNS, TARGET, *ROLE_COLUMNS, "final_role"
]
parquet_filter = [
    ("report_date", ">=", DEVELOPMENT_START),
    ("report_date", "<=", DEVELOPMENT_END),
]
locked_input_paths = [
    FINAL_PANEL_PATH, MEMBERSHIP_PATH, PERIOD_SUMMARY_PATH,
    MODEL_SELECTION_RULE_PATH, LOCKED_SETS_PATH,
    BANK_MANIFEST_PATH, FULL_MANIFEST_PATH, NB13_GATE_PATH,
]
input_hashes_before = {
    str(path): sha256_file(path) for path in locked_input_paths
}
CHECKPOINT_CONTEXT_SHA256 = hashlib.sha256(
    json.dumps(input_hashes_before, sort_keys=True).encode("utf-8")
).hexdigest()

panel = pd.read_parquet(
    FINAL_PANEL_PATH, columns=panel_columns, filters=parquet_filter
)
membership = pd.read_parquet(
    MEMBERSHIP_PATH, columns=membership_columns, filters=parquet_filter
)
for frame in [panel, membership]:
    frame["report_date"] = pd.to_datetime(frame["report_date"], errors="raise").dt.normalize()
    frame["cert"] = pd.to_numeric(frame["cert"], errors="raise").astype("int64")
    frame["idrssd"] = pd.to_numeric(frame["idrssd"], errors="raise").astype("int64")
    frame[TARGET] = pd.to_numeric(frame[TARGET], errors="raise").astype("int8")
panel = panel.loc[
    panel["report_date"].between(DEVELOPMENT_START, DEVELOPMENT_END, inclusive="both")
].copy()
membership = membership.loc[
    membership["report_date"].between(DEVELOPMENT_START, DEVELOPMENT_END, inclusive="both")
].copy()
for predictor in all_predictors:
    panel[predictor] = pd.to_numeric(panel[predictor], errors="coerce")

model_data = panel.merge(
    membership,
    on=IDENTIFIER_COLUMNS,
    how="inner",
    validate="one_to_one",
    suffixes=("", "_membership"),
)
target_matches = model_data[TARGET].eq(model_data[f"{TARGET}_membership"])
model_data = model_data.drop(columns=f"{TARGET}_membership")
development_use_mask = model_data[ROLE_COLUMNS].isin(["training", "validation"]).any(axis=1)
development_data = (
    model_data.loc[development_use_mask]
    .sort_values("panel_row_id", kind="mergesort")
    .reset_index(drop=True)
)

fold_reconciliation_rows = []
for item in VALIDATION_FOLDS:
    fold = f"fold{int(item['fold'])}"
    role_column = f"{fold}_role"
    for role in ["training", "validation"]:
        period = f"{fold}_{'train' if role == 'training' else 'validation'}"
        expected = period_summary.loc[period_summary["period"].eq(period)]
        scoped = development_data.loc[development_data[role_column].eq(role)]
        if len(expected) != 1:
            raise AssertionError(f"Notebook 10 period summary has {len(expected)} rows for {period}.")
        expected = expected.iloc[0]
        fold_reconciliation_rows.append({
            "fold": fold,
            "role": role,
            "expected_start": expected["start"],
            "observed_start": scoped["report_date"].min(),
            "expected_end": expected["end"],
            "observed_end": scoped["report_date"].max(),
            "expected_observations": int(expected["observations"]),
            "observed_observations": int(len(scoped)),
            "expected_positive_bank_quarters": int(expected["positive_bank_quarters"]),
            "observed_positive_bank_quarters": int(scoped[TARGET].sum()),
            "observed_negative_bank_quarters": int(scoped[TARGET].eq(0).sum()),
            "passed": (
                len(scoped) == int(expected["observations"])
                and int(scoped[TARGET].sum()) == int(expected["positive_bank_quarters"])
                and scoped["report_date"].min() == expected["start"]
                and scoped["report_date"].max() == expected["end"]
            ),
        })
fold_reconciliation = pd.DataFrame(fold_reconciliation_rows)

input_checks = {
    "Bank-only manifest is the exact ordered 19-feature lock":
        len(BANK_ONLY_PREDICTORS) == 19
        and BANK_ONLY_PREDICTORS == locked_sets["bank_only_features"],
    "Bank-plus-macro manifest is the exact ordered 23-feature lock":
        len(BANK_PLUS_MACRO_PREDICTORS) == 23
        and BANK_PLUS_MACRO_PREDICTORS == locked_sets["bank_plus_macro_features"],
    "History indicator is included in both specifications":
        all(HISTORY_INDICATOR in features for features in SPECIFICATIONS.values())
        and HISTORY_INDICATOR == "insufficient_history_5q_lag1",
    "No feature was added or removed in Notebook 11":
        locked_sets.get("additions") == [] and locked_sets.get("removals") == [],
    "Panel and membership development keys reconcile one-to-one":
        len(model_data) == len(panel) == len(membership)
        and not model_data.duplicated(IDENTIFIER_COLUMNS).any(),
    "Panel and membership targets match for every development row": target_matches.all(),
    "Development scope is exactly 2001Q2 through 2018Q4":
        development_data["report_date"].min() == DEVELOPMENT_START
        and development_data["report_date"].max() == DEVELOPMENT_END,
    "No 2019 purge or final-test row was loaded":
        not development_data["report_date"].gt(DEVELOPMENT_END).any()
        and development_data["final_role"].eq("training").all(),
    "Every fold count, date range and positive count reconciles to Notebook 10":
        fold_reconciliation["passed"].all(),
    "Every training and validation fold contains both classes":
        fold_reconciliation["observed_positive_bank_quarters"].gt(0).all()
        and fold_reconciliation["observed_negative_bank_quarters"].gt(0).all(),
    "All locked predictors are numeric and contain no infinity":
        all(pd.api.types.is_numeric_dtype(development_data[p]) for p in all_predictors)
        and not np.isinf(development_data[all_predictors].to_numpy(dtype="float64")).any(),
    "No identifier, date or target is in either feature set":
        set(BANK_PLUS_MACRO_PREDICTORS).isdisjoint(
            {TARGET, "panel_row_id", *IDENTIFIER_COLUMNS}
        ),
}
notebook_14_cell_02_gate = pd.DataFrame({
    "check": input_checks.keys(),
    "result": ["PASS" if value else "FAIL" for value in input_checks.values()],
    "passed": input_checks.values(),
})
fold_reconciliation.to_csv(NB14_OUTPUT_DIR / "cell_02_fold_reconciliation.csv", index=False)
notebook_14_cell_02_gate.to_csv(
    NB14_OUTPUT_DIR / "cell_02_input_reconciliation_gate.csv", index=False
)
pd.DataFrame([
    {"input_file": name, "sha256_before": digest}
    for name, digest in input_hashes_before.items()
]).to_csv(NB14_OUTPUT_DIR / "cell_02_input_hashes.csv", index=False)
print("Notebook 14 Cell 2 validation gate:")
display(notebook_14_cell_02_gate)
print("\nFold reconciliation:")
display(fold_reconciliation)
failed = [name for name, passed in input_checks.items() if not passed]
if failed:
    raise AssertionError("Cell 2 blocked: " + "; ".join(failed))
print("Checkpoint input context:", CHECKPOINT_CONTEXT_SHA256)


In [ ]:
# Define training-only preprocessing, metrics, selection and checkpoints

if not notebook_14_cell_02_gate["passed"].all():
    raise AssertionError("Cell 3 blocked because Cell 2 did not pass.")


def sha256_int64(values):
    array = np.ascontiguousarray(np.asarray(values, dtype="int64"))
    return hashlib.sha256(array.tobytes()).hexdigest()


def fit_training_preprocessor(training_frame, predictors):
    raw = training_frame[predictors].astype("float64")
    if np.isinf(raw.to_numpy()).any():
        raise AssertionError("Training predictors contain infinity.")
    medians = raw.median(axis=0, skipna=True)
    if medians.isna().any() or not np.isfinite(medians.to_numpy()).all():
        bad = medians.index[~np.isfinite(medians.to_numpy())].tolist()
        raise AssertionError("No finite training median for: " + ", ".join(bad))
    imputed = raw.fillna(medians)
    continuous = [name for name in predictors if name != HISTORY_INDICATOR]
    lower_bounds = imputed[continuous].quantile(
        WINSOR_LOWER_QUANTILE, interpolation="linear"
    )
    upper_bounds = imputed[continuous].quantile(
        WINSOR_UPPER_QUANTILE, interpolation="linear"
    )
    if (lower_bounds > upper_bounds).any():
        raise AssertionError("A lower winsor bound exceeds its upper bound.")
    imputed[continuous] = imputed[continuous].clip(
        lower=lower_bounds, upper=upper_bounds, axis=1
    )
    transformed = imputed[predictors].to_numpy(dtype="float32")
    if not np.isfinite(transformed).all():
        raise AssertionError("Training preprocessing produced a non-finite value.")
    fitted = {
        "predictors": list(predictors),
        "medians": medians,
        "continuous": continuous,
        "lower_bounds": lower_bounds,
        "upper_bounds": upper_bounds,
    }
    parameters = {
        "run_signature": RUN_SIGNATURE,
        "checkpoint_context_sha256": CHECKPOINT_CONTEXT_SHA256,
        "predictors": list(predictors),
        "fit_scope": "current specification and current training fold only",
        "imputation_strategy": "median",
        "missingness_covered": [
            "limited-history", "genuine", "FFIEC 051 structural"
        ],
        "imputation_medians": {name: float(medians[name]) for name in predictors},
        "winsor_lower_quantile": WINSOR_LOWER_QUANTILE,
        "winsor_upper_quantile": WINSOR_UPPER_QUANTILE,
        "winsor_quantile_interpolation": "linear",
        "winsor_excluded": [HISTORY_INDICATOR],
        "winsor_lower_bounds": {name: float(lower_bounds[name]) for name in continuous},
        "winsor_upper_bounds": {name: float(upper_bounds[name]) for name in continuous},
        "standardisation_applied": False,
        "native_missing_value_treatment_used": False,
        "all_training_values_finite_after_preprocessing": True,
        "validation_parameters_fitted": False,
        "test_data_used": False,
    }
    return transformed, fitted, parameters


def apply_training_preprocessor(other_frame, predictors, fitted):
    if list(predictors) != fitted["predictors"]:
        raise AssertionError("Predictor order differs from the fitted preprocessor.")
    raw = other_frame[predictors].astype("float64")
    if np.isinf(raw.to_numpy()).any():
        raise AssertionError("Validation predictors contain infinity.")
    imputed = raw.fillna(fitted["medians"])
    continuous = fitted["continuous"]
    imputed[continuous] = imputed[continuous].clip(
        lower=fitted["lower_bounds"], upper=fitted["upper_bounds"], axis=1
    )
    transformed = imputed[predictors].to_numpy(dtype="float32")
    if not np.isfinite(transformed).all():
        raise AssertionError("Validation preprocessing produced a non-finite value.")
    return transformed


def probability_metrics(y_true, probability, threshold=DEFAULT_CLASSIFICATION_THRESHOLD):
    predicted = (probability >= threshold).astype("int8")
    tn, fp, fn, tp = confusion_matrix(y_true, predicted, labels=[0, 1]).ravel()
    return {
        "roc_auc": float(roc_auc_score(y_true, probability)),
        "pr_auc": float(average_precision_score(y_true, probability)),
        "observed_prevalence": float(np.mean(y_true)),
        "mean_predicted_probability": float(np.mean(probability)),
        "threshold_for_descriptive_metrics": float(threshold),
        "recall_at_0p5": float(recall_score(y_true, predicted, zero_division=0)),
        "precision_at_0p5": float(precision_score(y_true, predicted, zero_division=0)),
        "f1_at_0p5": float(f1_score(y_true, predicted, zero_division=0)),
        "true_negatives_at_0p5": int(tn),
        "false_positives_at_0p5": int(fp),
        "false_negatives_at_0p5": int(fn),
        "true_positives_at_0p5": int(tp),
    }


def select_configuration(aggregate_table):
    candidates = aggregate_table.copy()
    best_pr = candidates["mean_pr_auc"].max()
    candidates = candidates.loc[
        candidates["mean_pr_auc"] >= best_pr - SELECTION_TOLERANCE
    ]
    best_roc = candidates["mean_roc_auc"].max()
    candidates = candidates.loc[
        candidates["mean_roc_auc"] >= best_roc - SELECTION_TOLERANCE
    ]
    best_std = candidates["std_pr_auc"].min()
    candidates = candidates.loc[
        candidates["std_pr_auc"] <= best_std + SELECTION_TOLERANCE
    ]
    return candidates.sort_values("config_id", kind="mergesort").iloc[0]


def write_json_atomic(payload, path):
    temporary = path.with_name(path.name + f".{SESSION_ID}.tmp")
    temporary.write_text(
        json.dumps(payload, indent=2, sort_keys=True, allow_nan=False),
        encoding="utf-8",
    )
    os.replace(temporary, path)


def write_parquet_atomic(frame, path):
    temporary = path.with_name(path.name + f".{SESSION_ID}.tmp")
    frame.to_parquet(temporary, index=False, compression="snappy")
    os.replace(temporary, path)


def load_valid_checkpoint(
    checkpoint_path, prediction_path, specification, fold, config_id,
    training_row_hash, validation_row_hash, expected_observations,
    expected_positives, preprocessing_sha256,
):
    if not checkpoint_path.is_file() or not prediction_path.is_file():
        return None
    try:
        payload = json.loads(checkpoint_path.read_text(encoding="utf-8"))
        identity_ok = (
            payload.get("run_signature") == RUN_SIGNATURE
            and payload.get("checkpoint_context_sha256") == CHECKPOINT_CONTEXT_SHA256
            and payload.get("specification") == specification
            and payload.get("fold") == fold
            and payload.get("config_id") == config_id
            and payload.get("training_panel_row_id_sha256") == training_row_hash
            and payload.get("validation_panel_row_id_sha256") == validation_row_hash
            and payload.get("preprocessing_sha256") == preprocessing_sha256
            and payload.get("prediction_file_sha256") == sha256_file(prediction_path)
        )
        if not identity_ok:
            return None
        predictions = pd.read_parquet(prediction_path)
        prediction_ok = (
            len(predictions) == expected_observations
            and int(predictions[TARGET].sum()) == expected_positives
            and not predictions["panel_row_id"].duplicated().any()
            and sha256_int64(predictions["panel_row_id"]) == validation_row_hash
            and np.isfinite(predictions["predicted_probability"]).all()
            and predictions["predicted_probability"].between(0, 1).all()
        )
        trial_row = payload.get("trial_metric_row", {})
        importance_rows = payload.get("importance_rows", [])
        manifest_row = payload.get("prediction_manifest_row", {})
        payload_ok = (
            prediction_ok
            and trial_row.get("config_id") == config_id
            and len(importance_rows) == len(SPECIFICATIONS[specification])
            and manifest_row.get("file") == str(prediction_path.relative_to(NB14_OUTPUT_DIR))
        )
        return payload if payload_ok else None
    except (OSError, ValueError, KeyError, TypeError):
        return None


implementation_contract = pd.DataFrame({
    "step": [
        "median imputation", "winsorisation", "class weighting",
        "row sampling", "feature sampling", "prediction", "checkpoint resume",
        "configuration selection",
    ],
    "fitted_or_derived_from": [
        "current training fold", "current training fold", "current training labels",
        "locked candidate applied to current training fold",
        "locked candidate applied to current training predictors",
        "current validation fold",
        "completed artifacts bound to current inputs and preprocessing",
        "three fold-level validation metrics",
    ],
    "validation_parameters_fitted": [False] * 8,
    "test_data_used": [False] * 8,
})
helper_checks = {
    "History indicator is excluded from winsorisation only":
        HISTORY_INDICATOR == "insufficient_history_5q_lag1",
    "All fitted preprocessing and weights are training-only":
        not implementation_contract["validation_parameters_fitted"].any(),
    "No helper loads or fits on the final test":
        not implementation_contract["test_data_used"].any(),
    "Standardisation is not applied to XGBoost inputs": True,
    "Native XGBoost missing treatment is structurally prevented":
        "median imputation" in implementation_contract["step"].tolist(),
    "Early stopping and threshold tuning are absent":
        EARLY_STOPPING_ROUNDS is None
        and "threshold selection" not in implementation_contract["step"].tolist(),
    "Checkpoint identity binds the grid, inputs, rows and preprocessing":
        len(RUN_SIGNATURE) > 20 and len(CHECKPOINT_CONTEXT_SHA256) == 64,
}
notebook_14_cell_03_gate = pd.DataFrame({
    "check": helper_checks.keys(),
    "result": ["PASS" if value else "FAIL" for value in helper_checks.values()],
    "passed": helper_checks.values(),
})
implementation_contract.to_csv(
    NB14_OUTPUT_DIR / "cell_03_implementation_contract.csv", index=False
)
notebook_14_cell_03_gate.to_csv(
    NB14_OUTPUT_DIR / "cell_03_helper_gate.csv", index=False
)
print("Notebook 14 Cell 3 validation gate:")
display(notebook_14_cell_03_gate)
display(implementation_contract)
failed = [name for name, passed in helper_checks.items() if not passed]
if failed:
    raise AssertionError("Cell 3 blocked: " + "; ".join(failed))


In [ ]:
# Fit or resume all 72 XGBoost fold trials

if not notebook_14_cell_03_gate["passed"].all():
    raise AssertionError("Cell 4 blocked because Cell 3 did not pass.")

trial_metric_rows = []
importance_rows_all = []
prediction_manifest_rows = []
preprocessing_manifest_rows = []
new_fits = 0
resumed_fits = 0
invalid_checkpoints_replaced = 0
session_start = time.perf_counter()

print("Atomic checkpoint/resume is active.")
print("A Colab reset can resume by rerunning the notebook from Cell 1.")

for specification, predictors in SPECIFICATIONS.items():
    print(f"\nRunning {specification} with {len(predictors)} predictors")
    for fold in FOLDS:
        role_column = f"{fold}_role"
        training = (
            development_data.loc[development_data[role_column].eq("training")]
            .sort_values("panel_row_id", kind="mergesort")
            .copy()
        )
        validation = (
            development_data.loc[development_data[role_column].eq("validation")]
            .sort_values("panel_row_id", kind="mergesort")
            .copy()
        )
        y_training = training[TARGET].to_numpy(dtype="int8")
        y_validation = validation[TARGET].to_numpy(dtype="int8")
        if set(np.unique(y_training)) != {0, 1} or set(np.unique(y_validation)) != {0, 1}:
            raise AssertionError(f"{specification} {fold} does not contain both classes.")

        x_training, fitted_preprocessor, preprocessing_parameters = (
            fit_training_preprocessor(training, predictors)
        )
        x_validation = apply_training_preprocessor(
            validation, predictors, fitted_preprocessor
        )
        training_row_hash = sha256_int64(training["panel_row_id"])
        validation_row_hash = sha256_int64(validation["panel_row_id"])
        preprocessing_parameters.update({
            "analysis_version": ANALYSIS_VERSION,
            "specification": specification,
            "fold": fold,
            "training_observations": int(len(training)),
            "validation_observations": int(len(validation)),
            "training_positives": int(y_training.sum()),
            "validation_positives": int(y_validation.sum()),
            "training_panel_row_id_sha256": training_row_hash,
            "validation_panel_row_id_sha256": validation_row_hash,
        })
        preprocessing_path = PREPROCESSING_DIR / f"{specification}__{fold}.json"
        write_json_atomic(preprocessing_parameters, preprocessing_path)
        preprocessing_sha256 = sha256_file(preprocessing_path)
        preprocessing_manifest_rows.append({
            "specification": specification,
            "fold": fold,
            "training_observations": int(len(training)),
            "validation_observations": int(len(validation)),
            "training_panel_row_id_sha256": training_row_hash,
            "validation_panel_row_id_sha256": validation_row_hash,
            "file": str(preprocessing_path.relative_to(NB14_OUTPUT_DIR)),
            "bytes": preprocessing_path.stat().st_size,
            "sha256": preprocessing_sha256,
        })

        negatives = int(np.sum(y_training == 0))
        positives = int(np.sum(y_training == 1))
        imbalance_ratio = negatives / positives

        for config in XGB_CONFIGURATIONS:
            config_id = config["config_id"]
            prediction_path = (
                PREDICTION_DIR / f"{specification}__{fold}__{config_id}.parquet"
            )
            checkpoint_path = (
                CHECKPOINT_DIR / f"{specification}__{fold}__{config_id}.json"
            )
            checkpoint_existed = checkpoint_path.is_file()
            checkpoint = load_valid_checkpoint(
                checkpoint_path=checkpoint_path,
                prediction_path=prediction_path,
                specification=specification,
                fold=fold,
                config_id=config_id,
                training_row_hash=training_row_hash,
                validation_row_hash=validation_row_hash,
                expected_observations=len(validation),
                expected_positives=int(y_validation.sum()),
                preprocessing_sha256=preprocessing_sha256,
            )
            if checkpoint is not None:
                trial_row = dict(checkpoint["trial_metric_row"])
                trial_row["resumed_from_checkpoint"] = True
                trial_row["current_session_id"] = SESSION_ID
                trial_metric_rows.append(trial_row)
                importance_rows_all.extend(checkpoint["importance_rows"])
                prediction_manifest_rows.append(checkpoint["prediction_manifest_row"])
                resumed_fits += 1
                print(f"  RESUME {fold} | {config_id} | PR-AUC={trial_row['pr_auc']:.6f}")
                continue
            if checkpoint_existed:
                invalid_checkpoints_replaced += 1

            positive_class_weight = float(
                imbalance_ratio * config["positive_weight_multiplier"]
            )
            fit_start = time.perf_counter()
            if not np.isfinite(x_training).all() or not np.isfinite(x_validation).all():
                raise AssertionError("Native missing-value handling was requested by non-finite input.")
            model = XGBClassifier(
                n_estimators=int(config["n_estimators"]),
                max_depth=int(config["max_depth"]),
                learning_rate=float(config["learning_rate"]),
                min_child_weight=float(config["min_child_weight"]),
                subsample=float(config["subsample"]),
                colsample_bytree=float(config["colsample_bytree"]),
                gamma=float(config["gamma"]),
                reg_lambda=float(config["reg_lambda"]),
                reg_alpha=float(config["reg_alpha"]),
                scale_pos_weight=positive_class_weight,
                objective=OBJECTIVE,
                eval_metric=XGB_EVAL_METRIC,
                tree_method=TREE_METHOD,
                importance_type="gain",
                random_state=RANDOM_SEED,
                n_jobs=N_JOBS,
                verbosity=0,
                validate_parameters=True,
            )
            model.fit(x_training, y_training)
            probability = model.predict_proba(x_validation)[:, 1]
            metrics = probability_metrics(y_validation, probability)
            fit_seconds = time.perf_counter() - fit_start

            trial_row = {
                "analysis_version": ANALYSIS_VERSION,
                "run_signature": RUN_SIGNATURE,
                "checkpoint_context_sha256": CHECKPOINT_CONTEXT_SHA256,
                "original_session_id": SESSION_ID,
                "current_session_id": SESSION_ID,
                "specification": specification,
                "fold": fold,
                **config,
                "xgboost_version": str(xgb.__version__),
                "tree_method": TREE_METHOD,
                "objective": OBJECTIVE,
                "eval_metric": XGB_EVAL_METRIC,
                "early_stopping_rounds": None,
                "native_missing_value_treatment_used": False,
                "training_imbalance_ratio": float(imbalance_ratio),
                "positive_class_weight": positive_class_weight,
                "training_observations": int(len(training)),
                "training_positives": positives,
                "validation_observations": int(len(validation)),
                "validation_positives": int(y_validation.sum()),
                "training_panel_row_id_sha256": training_row_hash,
                "validation_panel_row_id_sha256": validation_row_hash,
                "fit_seconds": float(fit_seconds),
                "resumed_from_checkpoint": False,
                **metrics,
            }
            fit_importance_rows = [
                {
                    "analysis_version": ANALYSIS_VERSION,
                    "run_signature": RUN_SIGNATURE,
                    "specification": specification,
                    "fold": fold,
                    "config_id": config_id,
                    "predictor": predictor,
                    "gain_importance": float(importance),
                }
                for predictor, importance in zip(predictors, model.feature_importances_)
            ]
            prediction_output = validation[
                ["panel_row_id", *IDENTIFIER_COLUMNS, TARGET]
            ].copy()
            prediction_output["analysis_version"] = ANALYSIS_VERSION
            prediction_output["run_signature"] = RUN_SIGNATURE
            prediction_output["specification"] = specification
            prediction_output["fold"] = fold
            prediction_output["config_id"] = config_id
            prediction_output["predicted_probability"] = probability.astype("float64")
            write_parquet_atomic(prediction_output, prediction_path)
            prediction_file_sha256 = sha256_file(prediction_path)
            prediction_manifest_row = {
                "specification": specification,
                "fold": fold,
                "config_id": config_id,
                "observations": int(len(prediction_output)),
                "positive_bank_quarters": int(y_validation.sum()),
                "panel_row_id_sha256": validation_row_hash,
                "file": str(prediction_path.relative_to(NB14_OUTPUT_DIR)),
                "bytes": prediction_path.stat().st_size,
                "sha256": prediction_file_sha256,
            }
            checkpoint_payload = {
                "run_signature": RUN_SIGNATURE,
                "checkpoint_context_sha256": CHECKPOINT_CONTEXT_SHA256,
                "specification": specification,
                "fold": fold,
                "config_id": config_id,
                "training_panel_row_id_sha256": training_row_hash,
                "validation_panel_row_id_sha256": validation_row_hash,
                "preprocessing_sha256": preprocessing_sha256,
                "prediction_file_sha256": prediction_file_sha256,
                "trial_metric_row": trial_row,
                "importance_rows": fit_importance_rows,
                "prediction_manifest_row": prediction_manifest_row,
            }
            write_json_atomic(checkpoint_payload, checkpoint_path)
            trial_metric_rows.append(trial_row)
            importance_rows_all.extend(fit_importance_rows)
            prediction_manifest_rows.append(prediction_manifest_row)
            new_fits += 1
            print(
                f"  FIT {fold} | {config_id} | PR-AUC={metrics['pr_auc']:.6f} | "
                f"ROC-AUC={metrics['roc_auc']:.6f} | {fit_seconds:.1f}s"
            )
            del model, probability, prediction_output
            gc.collect()

        del x_training, x_validation, fitted_preprocessor, training, validation
        gc.collect()

trial_metrics = pd.DataFrame(trial_metric_rows)
importance_register = pd.DataFrame(importance_rows_all)
prediction_manifest = pd.DataFrame(prediction_manifest_rows)
preprocessing_manifest = pd.DataFrame(preprocessing_manifest_rows)
total_session_seconds = time.perf_counter() - session_start
expected_checkpoint_paths = [
    CHECKPOINT_DIR / f"{specification}__{fold}__{config['config_id']}.json"
    for specification in SPECIFICATIONS
    for fold in FOLDS
    for config in XGB_CONFIGURATIONS
]
expected_fits = len(SPECIFICATIONS) * len(FOLDS) * len(XGB_CONFIGURATIONS)
current_grid_reconciliation = (
    trial_metrics[[
        "config_id", "n_estimators", "max_depth", "learning_rate",
        "min_child_weight", "subsample", "colsample_bytree", "gamma",
        "reg_lambda", "reg_alpha", "positive_weight_multiplier",
    ]]
    .drop_duplicates()
    .sort_values("config_id")
    .reset_index(drop=True)
    .astype(str)
    .equals(
        config_frame[[
            "config_id", "n_estimators", "max_depth", "learning_rate",
            "min_child_weight", "subsample", "colsample_bytree", "gamma",
            "reg_lambda", "reg_alpha", "positive_weight_multiplier",
        ]]
        .sort_values("config_id")
        .reset_index(drop=True)
        .astype(str)
    )
)
development_checks = {
    "Exactly 72 fold-level fits were completed or resumed":
        len(trial_metrics) == expected_fits == 72
        and new_fits + resumed_fits == 72,
    "Every specification-fold contains the exact 12-configuration grid":
        trial_metrics.groupby(["specification", "fold"])["config_id"].nunique().eq(12).all()
        and current_grid_reconciliation,
    "Every result is bound to the current run and input context":
        trial_metrics["run_signature"].eq(RUN_SIGNATURE).all()
        and trial_metrics["checkpoint_context_sha256"].eq(
            CHECKPOINT_CONTEXT_SHA256
        ).all(),
    "Every fit used the locked histogram method and no native missing treatment":
        trial_metrics["tree_method"].eq(TREE_METHOD).all()
        and not trial_metrics["native_missing_value_treatment_used"].any(),
    "All PR-AUC and ROC-AUC values are finite and bounded":
        np.isfinite(trial_metrics[["pr_auc", "roc_auc"]].to_numpy()).all()
        and trial_metrics["pr_auc"].between(0, 1).all()
        and trial_metrics["roc_auc"].between(0, 1).all(),
    "All 72 predictions and checkpoints exist":
        len(prediction_manifest) == prediction_manifest["file"].nunique() == 72
        and all((NB14_OUTPUT_DIR / path).is_file() for path in prediction_manifest["file"])
        and all(path.is_file() for path in expected_checkpoint_paths),
    "Six training-only preprocessing files were written":
        len(preprocessing_manifest) == preprocessing_manifest["file"].nunique() == 6
        and all((NB14_OUTPUT_DIR / path).is_file() for path in preprocessing_manifest["file"]),
    "Feature importance covers every predictor in every fit":
        len(importance_register) == sum(
            len(SPECIFICATIONS[specification])
            for specification in SPECIFICATIONS
            for _fold in FOLDS
            for _config in XGB_CONFIGURATIONS
        ),
}
notebook_14_cell_04_gate = pd.DataFrame({
    "check": development_checks.keys(),
    "result": ["PASS" if value else "FAIL" for value in development_checks.values()],
    "passed": development_checks.values(),
})
trial_metrics.to_csv(
    NB14_OUTPUT_DIR / "all_xgboost_trial_metrics.csv", index=False
)
importance_register.to_csv(
    NB14_OUTPUT_DIR / "all_xgboost_feature_importances.csv", index=False
)
prediction_manifest.to_csv(
    NB14_OUTPUT_DIR / "validation_prediction_manifest.csv", index=False
)
preprocessing_manifest.to_csv(
    NB14_OUTPUT_DIR / "preprocessing_parameter_manifest.csv", index=False
)
pd.DataFrame([{
    "session_id": SESSION_ID,
    "run_signature": RUN_SIGNATURE,
    "new_fits": new_fits,
    "resumed_fits": resumed_fits,
    "invalid_checkpoints_replaced": invalid_checkpoints_replaced,
    "session_runtime_seconds": total_session_seconds,
}]).to_csv(NB14_OUTPUT_DIR / "cell_04_session_resume_summary.csv", index=False)
notebook_14_cell_04_gate.to_csv(
    NB14_OUTPUT_DIR / "cell_04_model_development_gate.csv", index=False
)
print("\nNotebook 14 Cell 4 validation gate:")
display(notebook_14_cell_04_gate)
print(
    f"\nSession result: {new_fits} newly fitted, {resumed_fits} resumed, "
    f"{invalid_checkpoints_replaced} invalid checkpoint(s) safely replaced."
)
print(f"Session runtime: {total_session_seconds / 60:.2f} minutes")
failed = [name for name, passed in development_checks.items() if not passed]
if failed:
    raise AssertionError("Cell 4 blocked: " + "; ".join(failed))


In [ ]:
# Aggregate folds and select one V2 XGBoost configuration per specification

if not notebook_14_cell_04_gate["passed"].all():
    raise AssertionError("Cell 5 blocked because Cell 4 did not pass.")

metric_aggregates = (
    trial_metrics.groupby(["specification", "config_id"], as_index=False)
    .agg(
        folds=("fold", "nunique"),
        mean_pr_auc=("pr_auc", "mean"),
        std_pr_auc=("pr_auc", "std"),
        min_pr_auc=("pr_auc", "min"),
        max_pr_auc=("pr_auc", "max"),
        mean_roc_auc=("roc_auc", "mean"),
        std_roc_auc=("roc_auc", "std"),
        mean_recall_at_0p5=("recall_at_0p5", "mean"),
        mean_precision_at_0p5=("precision_at_0p5", "mean"),
        mean_f1_at_0p5=("f1_at_0p5", "mean"),
        cumulative_fit_seconds=("fit_seconds", "sum"),
    )
)
aggregate_results = metric_aggregates.merge(
    config_frame,
    on="config_id",
    how="left",
    validate="many_to_one",
)
aggregate_results["tree_method"] = TREE_METHOD
aggregate_results["native_missing_value_treatment_used"] = False

selected_rows = []
for specification in SPECIFICATIONS:
    selected_rows.append(
        select_configuration(
            aggregate_results.loc[
                aggregate_results["specification"].eq(specification)
            ].copy()
        ).to_dict()
    )
selected_configurations = pd.DataFrame(selected_rows)
selected_configurations.insert(0, "analysis_version", ANALYSIS_VERSION)
selected_configurations.insert(1, "run_signature", RUN_SIGNATURE)
selected_configurations["selection_rule"] = (
    "highest unweighted mean validation PR-AUC; within 1e-6, higher mean "
    "validation ROC-AUC; within 1e-6 again, lower validation PR-AUC standard deviation"
)
selected_configurations["threshold_selected"] = False
selected_configurations["threshold_selection_deferred_to"] = (
    "15_Validation_Comparison_and_Model_Lock.ipynb"
)
selected_configurations["final_test_used"] = False
selected_configurations["version_1_winner_reused"] = False

selected_fold_metrics = trial_metrics.merge(
    selected_configurations[["specification", "config_id"]],
    on=["specification", "config_id"],
    how="inner",
    validate="many_to_one",
)
selected_importances = importance_register.merge(
    selected_configurations[["specification", "config_id"]],
    on=["specification", "config_id"],
    how="inner",
    validate="many_to_one",
)
selection_checks = {
    "Every configuration is aggregated over exactly three folds":
        aggregate_results["folds"].eq(3).all(),
    "Exactly 12 aggregate configurations exist per specification":
        aggregate_results.groupby("specification")["config_id"].nunique().eq(12).all(),
    "Exactly one configuration is selected per specification":
        len(selected_configurations) == 2
        and selected_configurations["specification"].nunique() == 2,
    "Each selected configuration has three fold metrics":
        selected_fold_metrics.groupby("specification")["fold"].nunique().eq(3).all(),
    "Selected importance contains every selected fit and predictor":
        selected_importances.groupby(["specification", "fold"])["predictor"].nunique().to_dict()
        == {
            (specification, fold): len(SPECIFICATIONS[specification])
            for specification in SPECIFICATIONS
            for fold in FOLDS
        },
    "No threshold, final test or Version 1 winner informs selection":
        not selected_configurations["threshold_selected"].any()
        and not selected_configurations["final_test_used"].any()
        and not selected_configurations["version_1_winner_reused"].any(),
}
notebook_14_cell_05_gate = pd.DataFrame({
    "check": selection_checks.keys(),
    "result": ["PASS" if value else "FAIL" for value in selection_checks.values()],
    "passed": selection_checks.values(),
})
aggregate_results.to_csv(
    NB14_OUTPUT_DIR / "xgboost_aggregate_validation_results.csv", index=False
)
selected_configurations.to_csv(
    NB14_OUTPUT_DIR / "selected_xgboost_configurations.csv", index=False
)
with open(
    NB14_OUTPUT_DIR / "selected_xgboost_configurations.json",
    "w", encoding="utf-8",
) as handle:
    json.dump(
        json.loads(selected_configurations.to_json(orient="records")),
        handle, indent=2, allow_nan=False,
    )
selected_fold_metrics.to_csv(
    NB14_OUTPUT_DIR / "selected_xgboost_fold_metrics.csv", index=False
)
selected_importances.to_csv(
    NB14_OUTPUT_DIR / "selected_xgboost_feature_importances.csv", index=False
)
notebook_14_cell_05_gate.to_csv(
    NB14_OUTPUT_DIR / "cell_05_selection_gate.csv", index=False
)
print("Notebook 14 Cell 5 validation gate:")
display(notebook_14_cell_05_gate)
print("\nSelected Version 2 XGBoost configurations:")
with pd.option_context("display.max_colwidth", None):
    display(selected_configurations)
failed = [name for name, passed in selection_checks.items() if not passed]
if failed:
    raise AssertionError("Cell 5 blocked: " + "; ".join(failed))


In [ ]:
# Re-read predictions, preprocessing and checkpoints and recompute metrics

if not notebook_14_cell_05_gate["passed"].all():
    raise AssertionError("Cell 6 blocked because Cell 5 did not pass.")

reloaded_metrics = pd.read_csv(NB14_OUTPUT_DIR / "all_xgboost_trial_metrics.csv")
reloaded_aggregate = pd.read_csv(
    NB14_OUTPUT_DIR / "xgboost_aggregate_validation_results.csv"
)
reloaded_selected = pd.read_csv(
    NB14_OUTPUT_DIR / "selected_xgboost_configurations.csv"
)
reloaded_prediction_manifest = pd.read_csv(
    NB14_OUTPUT_DIR / "validation_prediction_manifest.csv"
)
reloaded_preprocessing_manifest = pd.read_csv(
    NB14_OUTPUT_DIR / "preprocessing_parameter_manifest.csv"
)
reloaded_importances = pd.read_csv(
    NB14_OUTPUT_DIR / "all_xgboost_feature_importances.csv"
)
with open(
    NB14_OUTPUT_DIR / "selected_xgboost_configurations.json",
    "r", encoding="utf-8",
) as handle:
    reloaded_selected_json = json.load(handle)

metric_lookup = reloaded_metrics.set_index(["specification", "fold", "config_id"])
prediction_audit_rows = []
for row in reloaded_prediction_manifest.itertuples(index=False):
    path = NB14_OUTPUT_DIR / row.file
    predictions = pd.read_parquet(path)
    saved_metric = metric_lookup.loc[(row.specification, row.fold, row.config_id)]
    recomputed_pr = average_precision_score(
        predictions[TARGET], predictions["predicted_probability"]
    )
    recomputed_roc = roc_auc_score(
        predictions[TARGET], predictions["predicted_probability"]
    )
    expected_validation = development_data.loc[
        development_data[f"{row.fold}_role"].eq("validation")
    ].sort_values("panel_row_id", kind="mergesort")
    prediction_audit_rows.append({
        "specification": row.specification,
        "fold": row.fold,
        "config_id": row.config_id,
        "file": row.file,
        "file_sha256_matches_manifest": sha256_file(path) == row.sha256,
        "observations": int(len(predictions)),
        "expected_observations": int(len(expected_validation)),
        "positive_bank_quarters": int(predictions[TARGET].sum()),
        "expected_positive_bank_quarters": int(expected_validation[TARGET].sum()),
        "panel_row_id_sha256": sha256_int64(predictions["panel_row_id"]),
        "expected_panel_row_id_sha256": sha256_int64(expected_validation["panel_row_id"]),
        "probability_min": float(predictions["predicted_probability"].min()),
        "probability_max": float(predictions["predicted_probability"].max()),
        "duplicate_panel_row_ids": int(predictions["panel_row_id"].duplicated().sum()),
        "maximum_reference_date": predictions["report_date"].max(),
        "pr_auc_recomputed": float(recomputed_pr),
        "pr_auc_saved": float(saved_metric["pr_auc"]),
        "roc_auc_recomputed": float(recomputed_roc),
        "roc_auc_saved": float(saved_metric["roc_auc"]),
        "metric_match": (
            np.isclose(recomputed_pr, saved_metric["pr_auc"], rtol=0.0, atol=1e-12)
            and np.isclose(recomputed_roc, saved_metric["roc_auc"], rtol=0.0, atol=1e-12)
        ),
    })
prediction_audit = pd.DataFrame(prediction_audit_rows)

preprocessing_audit_rows = []
for row in reloaded_preprocessing_manifest.itertuples(index=False):
    path = NB14_OUTPUT_DIR / row.file
    parameters = json.loads(path.read_text(encoding="utf-8"))
    predictors = SPECIFICATIONS[row.specification]
    continuous = [name for name in predictors if name != HISTORY_INDICATOR]
    numerical_values = [
        *parameters["imputation_medians"].values(),
        *parameters["winsor_lower_bounds"].values(),
        *parameters["winsor_upper_bounds"].values(),
    ]
    preprocessing_audit_rows.append({
        "specification": row.specification,
        "fold": row.fold,
        "file": row.file,
        "file_sha256_matches_manifest": sha256_file(path) == row.sha256,
        "predictor_order_matches_lock": parameters["predictors"] == predictors,
        "all_predictors_have_medians":
            set(parameters["imputation_medians"]) == set(predictors),
        "all_continuous_predictors_have_winsor_bounds":
            set(parameters["winsor_lower_bounds"]) == set(continuous)
            and set(parameters["winsor_upper_bounds"]) == set(continuous),
        "history_indicator_excluded_only_from_winsorisation":
            parameters["winsor_excluded"] == [HISTORY_INDICATOR]
            and HISTORY_INDICATOR in parameters["imputation_medians"],
        "all_parameters_finite": bool(np.isfinite(numerical_values).all()),
        "native_missing_value_treatment_used":
            parameters["native_missing_value_treatment_used"],
        "standardisation_applied": parameters["standardisation_applied"],
        "validation_parameters_fitted": parameters["validation_parameters_fitted"],
        "test_data_used": parameters["test_data_used"],
    })
preprocessing_audit = pd.DataFrame(preprocessing_audit_rows)

checkpoint_audit_rows = []
for specification in SPECIFICATIONS:
    for fold in FOLDS:
        for config in XGB_CONFIGURATIONS:
            config_id = config["config_id"]
            path = CHECKPOINT_DIR / f"{specification}__{fold}__{config_id}.json"
            payload = json.loads(path.read_text(encoding="utf-8"))
            checkpoint_audit_rows.append({
                "specification": specification,
                "fold": fold,
                "config_id": config_id,
                "file": str(path.relative_to(NB14_OUTPUT_DIR)),
                "run_signature_matches": payload["run_signature"] == RUN_SIGNATURE,
                "input_context_matches":
                    payload["checkpoint_context_sha256"] == CHECKPOINT_CONTEXT_SHA256,
                "prediction_hash_matches":
                    payload["prediction_file_sha256"] == sha256_file(
                        PREDICTION_DIR / f"{specification}__{fold}__{config_id}.parquet"
                    ),
            })
checkpoint_audit = pd.DataFrame(checkpoint_audit_rows)

importance_sums = (
    reloaded_importances.groupby(["specification", "fold", "config_id"])
    .agg(
        predictors=("predictor", "nunique"),
        importance_sum=("gain_importance", "sum"),
        minimum_importance=("gain_importance", "min"),
    )
    .reset_index()
)
importance_sums["no_split_booster"] = np.isclose(
    importance_sums["importance_sum"], 0.0, atol=1e-12
)
importance_sums["valid_importance_sum"] = (
    np.isclose(importance_sums["importance_sum"], 1.0, atol=1e-10)
    | importance_sums["no_split_booster"]
)
expected_predictors = importance_sums["specification"].map(
    {name: len(features) for name, features in SPECIFICATIONS.items()}
)
reread_checks = {
    "All 72 metrics, 24 aggregates and two selections re-read":
        len(reloaded_metrics) == 72
        and len(reloaded_aggregate) == 24
        and len(reloaded_selected) == len(reloaded_selected_json) == 2,
    "All 72 prediction files match hashes and complete validation folds":
        len(prediction_audit) == 72
        and prediction_audit["file_sha256_matches_manifest"].all()
        and prediction_audit["observations"].eq(
            prediction_audit["expected_observations"]
        ).all()
        and prediction_audit["positive_bank_quarters"].eq(
            prediction_audit["expected_positive_bank_quarters"]
        ).all()
        and prediction_audit["panel_row_id_sha256"].eq(
            prediction_audit["expected_panel_row_id_sha256"]
        ).all(),
    "All probabilities are finite, bounded and development-only":
        np.isfinite(prediction_audit[["probability_min", "probability_max"]].to_numpy()).all()
        and prediction_audit["probability_min"].ge(0).all()
        and prediction_audit["probability_max"].le(1).all()
        and prediction_audit["duplicate_panel_row_ids"].eq(0).all()
        and pd.to_datetime(prediction_audit["maximum_reference_date"]).le(
            DEVELOPMENT_END
        ).all(),
    "Every saved PR-AUC and ROC-AUC recomputes within tolerance":
        prediction_audit["metric_match"].all(),
    "All six preprocessing files match the training-only lock":
        len(preprocessing_audit) == 6
        and preprocessing_audit[[
            "file_sha256_matches_manifest", "predictor_order_matches_lock",
            "all_predictors_have_medians",
            "all_continuous_predictors_have_winsor_bounds",
            "history_indicator_excluded_only_from_winsorisation",
            "all_parameters_finite",
        ]].all().all()
        and not preprocessing_audit["standardisation_applied"].any()
        and not preprocessing_audit["native_missing_value_treatment_used"].any()
        and not preprocessing_audit["validation_parameters_fitted"].any()
        and not preprocessing_audit["test_data_used"].any(),
    "All 72 checkpoints are bound to current inputs and predictions":
        len(checkpoint_audit) == 72
        and checkpoint_audit[[
            "run_signature_matches", "input_context_matches", "prediction_hash_matches"
        ]].all().all(),
    "Every feature-importance vector is complete, finite and correctly normalised":
        importance_sums["predictors"].eq(expected_predictors).all()
        and np.isfinite(reloaded_importances["gain_importance"]).all()
        and importance_sums["minimum_importance"].ge(0).all()
        and importance_sums["valid_importance_sum"].all(),
}
notebook_14_cell_06_gate = pd.DataFrame({
    "check": reread_checks.keys(),
    "result": ["PASS" if value else "FAIL" for value in reread_checks.values()],
    "passed": reread_checks.values(),
})
prediction_audit.to_csv(
    NB14_OUTPUT_DIR / "cell_06_prediction_coverage_audit.csv", index=False
)
preprocessing_audit.to_csv(
    NB14_OUTPUT_DIR / "cell_06_preprocessing_parameter_audit.csv", index=False
)
checkpoint_audit.to_csv(
    NB14_OUTPUT_DIR / "cell_06_checkpoint_audit.csv", index=False
)
importance_sums.to_csv(
    NB14_OUTPUT_DIR / "cell_06_feature_importance_audit.csv", index=False
)
notebook_14_cell_06_gate.to_csv(
    NB14_OUTPUT_DIR / "cell_06_handoff_audit_gate.csv", index=False
)
print("Notebook 14 Cell 6 validation gate:")
display(notebook_14_cell_06_gate)
failed = [name for name, passed in reread_checks.items() if not passed]
if failed:
    raise AssertionError("Cell 6 blocked: " + "; ".join(failed))


In [ ]:
# Prove input immutability and save the final Notebook 14 handoff

if not notebook_14_cell_06_gate["passed"].all():
    raise AssertionError("Cell 7 blocked because Cell 6 did not pass.")

input_hashes_after = {str(path): sha256_file(path) for path in locked_input_paths}
input_immutability = pd.DataFrame([
    {
        "input_file": name,
        "sha256_before": input_hashes_before[name],
        "sha256_after": input_hashes_after[name],
        "unchanged": input_hashes_before[name] == input_hashes_after[name],
    }
    for name in input_hashes_before
])
input_immutability.to_csv(
    NB14_OUTPUT_DIR / "cell_07_input_immutability_audit.csv", index=False
)
cumulative_fit_seconds = float(trial_metrics["fit_seconds"].sum())
notebook_14_final_summary = pd.DataFrame({
    "measure": [
        "Analysis version", "Model family", "Specifications", "Validation folds",
        "Development reference period", "Configurations per specification",
        "Total fold-level fits", "Number-of-trees values", "Maximum-depth values",
        "Learning-rate values", "Minimum-child-weight values", "Row-sampling values",
        "Feature-sampling values", "Gamma values", "L2-regularisation values",
        "L1-regularisation values", "Weight multipliers", "Tree method",
        "Checkpoint/resume enabled",
        "Fits resumed this session", "Fits newly trained this session",
        "Invalid checkpoints safely replaced", "Bank-only predictors",
        "Bank-plus-macro predictors", "History indicator in both specifications",
        "Training-only imputation", "Training-only winsorisation",
        "Native missing-value treatment used", "Standardisation applied",
        "Early stopping used", "Validation prediction files",
        "Preprocessing parameter files", "Atomic fit checkpoints",
        "Selected configurations", "Thresholds selected", "Final-test rows loaded",
        "Version 1 winners or fitted artifacts reused", "Input files modified",
        "Session runtime minutes", "Cumulative recorded fit minutes",
        "Next modelling notebook",
    ],
    "value": [
        ANALYSIS_VERSION, "XGBoost", 2, 3,
        f"{DEVELOPMENT_START.to_period('Q')}–{DEVELOPMENT_END.to_period('Q')}",
        len(XGB_CONFIGURATIONS), len(trial_metrics),
        str(sorted(config_frame["n_estimators"].unique().tolist())),
        str(sorted(config_frame["max_depth"].unique().tolist())),
        str(sorted(config_frame["learning_rate"].unique().tolist())),
        str(sorted(config_frame["min_child_weight"].unique().tolist())),
        str(sorted(config_frame["subsample"].unique().tolist())),
        str(sorted(config_frame["colsample_bytree"].unique().tolist())),
        str(sorted(config_frame["gamma"].unique().tolist())),
        str(sorted(config_frame["reg_lambda"].unique().tolist())),
        str(sorted(config_frame["reg_alpha"].unique().tolist())),
        str(sorted(config_frame["positive_weight_multiplier"].unique().tolist())),
        TREE_METHOD, True, resumed_fits, new_fits, invalid_checkpoints_replaced,
        len(BANK_ONLY_PREDICTORS), len(BANK_PLUS_MACRO_PREDICTORS), True,
        True, True, False, False, False, len(prediction_manifest), len(preprocessing_manifest),
        len(checkpoint_audit), len(selected_configurations), 0, 0, 0,
        int((~input_immutability["unchanged"]).sum()),
        total_session_seconds / 60, cumulative_fit_seconds / 60,
        "15_Validation_Comparison_and_Model_Lock.ipynb",
    ],
})
notebook_14_final_summary.to_csv(
    NB14_OUTPUT_DIR / "cell_07_final_summary.csv", index=False
)
earlier_gates = [
    notebook_14_cell_01_gate, notebook_14_cell_02_gate,
    notebook_14_cell_03_gate, notebook_14_cell_04_gate,
    notebook_14_cell_05_gate, notebook_14_cell_06_gate,
]
final_checks = {
    "All earlier Notebook 14 gates passed":
        all(gate["passed"].all() for gate in earlier_gates),
    "Both locked XGBoost specifications were developed":
        set(trial_metrics["specification"]) == {"bank_only", "bank_plus_macro"},
    "All folds, configurations, metrics, predictions and checkpoints were saved":
        len(trial_metrics) == len(prediction_manifest) == len(checkpoint_audit) == 72
        and trial_metrics.groupby("specification")["fold"].nunique().eq(3).all(),
    "One configuration was selected for each specification":
        selected_configurations["specification"].nunique() == 2,
    "All six training-only preprocessing parameter sets were saved":
        len(preprocessing_manifest) == 6,
    "Checkpoint resume preserved or completed every required fit":
        resumed_fits + new_fits == 72,
    "All predictor missingness was training-median-imputed before fitting":
        not preprocessing_audit["native_missing_value_treatment_used"].any()
        and preprocessing_audit["all_predictors_have_medians"].all(),
    "No early stopping was used": EARLY_STOPPING_ROUNDS is None,
    "No threshold was tuned or selected":
        not selected_configurations["threshold_selected"].any(),
    "The final test remained untouched":
        not selected_configurations["final_test_used"].any()
        and pd.to_datetime(prediction_audit["maximum_reference_date"]).max()
        <= DEVELOPMENT_END,
    "No Version 1 winner or fitted preprocessing was reused":
        not selected_configurations["version_1_winner_reused"].any(),
    "Every locked input remained byte-for-byte unchanged":
        input_immutability["unchanged"].all(),
}
notebook_14_final_validation_gate = pd.DataFrame({
    "check": final_checks.keys(),
    "result": ["PASS" if value else "FAIL" for value in final_checks.values()],
    "passed": final_checks.values(),
})
notebook_14_final_validation_gate.to_csv(
    NB14_OUTPUT_DIR / "cell_07_final_validation_gate.csv", index=False
)

principal_purposes = {
    "cell_01_xgboost_configuration_grid.csv": "Unchanged completed-V1 XGBoost candidate grid",
    "all_xgboost_trial_metrics.csv": "Every specification, fold, configuration and metric",
    "xgboost_aggregate_validation_results.csv": "Unweighted three-fold validation aggregation",
    "selected_xgboost_configurations.csv": "One selected V2 configuration per specification",
    "selected_xgboost_configurations.json": "Machine-readable selected-configuration handoff",
    "selected_xgboost_fold_metrics.csv": "Three fold metrics for each selected configuration",
    "all_xgboost_feature_importances.csv": "Gain importance for every fold fit",
    "selected_xgboost_feature_importances.csv": "Importance rows for selected configurations",
    "validation_prediction_manifest.csv": "Index and hashes of all 72 validation prediction files",
    "preprocessing_parameter_manifest.csv": "Index and hashes of six training-fold preprocessing files",
    "cell_06_prediction_coverage_audit.csv": "Prediction coverage and metric recomputation audit",
    "cell_06_preprocessing_parameter_audit.csv": "Training-only preprocessing re-read audit",
    "cell_06_checkpoint_audit.csv": "Checkpoint binding and prediction-hash audit",
    "cell_06_feature_importance_audit.csv": "Completeness and normalisation audit; zero vectors identify no-split boosters",
    "cell_07_input_immutability_audit.csv": "Before-and-after hashes of locked inputs",
    "cell_07_final_summary.csv": "Notebook 14 final scope and treatment summary",
    "cell_07_final_validation_gate.csv": "Final fail-fast validation gate",
}
principal_paths = [NB14_OUTPUT_DIR / name for name in principal_purposes]
detail_paths = [
    *[NB14_OUTPUT_DIR / path for path in prediction_manifest["file"]],
    *[NB14_OUTPUT_DIR / path for path in preprocessing_manifest["file"]],
    *[
        CHECKPOINT_DIR / f"{specification}__{fold}__{config['config_id']}.json"
        for specification in SPECIFICATIONS
        for fold in FOLDS
        for config in XGB_CONFIGURATIONS
    ],
    SESSION_DIR / f"{SESSION_ID}.json",
]
audit_paths = [
    NB14_OUTPUT_DIR / "cell_01_setup_gate.csv",
    NB14_OUTPUT_DIR / "cell_01_xgboost_tuning_contract.csv",
    NB14_OUTPUT_DIR / "cell_02_fold_reconciliation.csv",
    NB14_OUTPUT_DIR / "cell_02_input_reconciliation_gate.csv",
    NB14_OUTPUT_DIR / "cell_02_input_hashes.csv",
    NB14_OUTPUT_DIR / "cell_03_implementation_contract.csv",
    NB14_OUTPUT_DIR / "cell_03_helper_gate.csv",
    NB14_OUTPUT_DIR / "cell_04_session_resume_summary.csv",
    NB14_OUTPUT_DIR / "cell_04_model_development_gate.csv",
    NB14_OUTPUT_DIR / "cell_05_selection_gate.csv",
    NB14_OUTPUT_DIR / "cell_06_handoff_audit_gate.csv",
]
all_manifest_paths = []
for path in [*audit_paths, *principal_paths, *detail_paths]:
    if path not in all_manifest_paths:
        all_manifest_paths.append(path)
missing_outputs = [path for path in all_manifest_paths if not path.is_file()]
if missing_outputs:
    raise AssertionError("Missing Notebook 14 outputs: " + "; ".join(map(str, missing_outputs)))
output_manifest = pd.DataFrame([
    {
        "file": str(path.relative_to(NB14_OUTPUT_DIR)),
        "purpose": principal_purposes.get(
            path.name, "Gate, audit, checkpoint, or detailed current-run artifact"
        ),
        "bytes": path.stat().st_size,
        "sha256": sha256_file(path),
        "run_signature": RUN_SIGNATURE,
        "session_id": SESSION_ID,
    }
    for path in all_manifest_paths
])
output_manifest.to_csv(NB14_OUTPUT_DIR / "cell_07_output_manifest.csv", index=False)

print("Notebook 14 Cell 7 final validation gate:")
display(notebook_14_final_validation_gate)
print("\nNotebook 14 final summary:")
display(notebook_14_final_summary)
print("\nSelected Version 2 XGBoost configurations:")
display(selected_configurations)
print("\nOutput manifest:")
display(output_manifest)
failed = [name for name, passed in final_checks.items() if not passed]
if failed:
    raise AssertionError("Cell 7 blocked: " + "; ".join(failed))
if not (NB14_OUTPUT_DIR / "cell_07_output_manifest.csv").is_file():
    raise AssertionError("Notebook 14 output manifest was not written.")
print("\nNotebook 14 is PASS / LOCKED.")
print("Selected configuration handoff:")
print(NB14_OUTPUT_DIR / "selected_xgboost_configurations.json")
print("Notebook 15 may now compare and lock all six validation-developed specifications.")
